In [1]:
import pandas as pd
import numpy as np

# Checking the inputs produced (data.csv)
df = pd.read_csv('data.csv')
print("Initial Data Shape:", df.shape)
df.head()


Initial Data Shape: (2599, 19)


,Date,Symbol,Futures Contracts,Futures Quantity,Futures Value (Rs. In Crs.),Options Contracts,Options Quantity,Options Value (Rs. In Crs.),Total Overall Value (Rs. In Crs.),Equity Traded Value (Rs. In Crs.),Speculation Ratio,Equity Traded Quantity (in Lakhs),Equity Number of Trades,Equity Total Market Cap (Rs. Crs.),options/futures,open,high,low,close
0,2016-01-01,NIFTY,81885,6141375,4890.54,859166,64437450,52040.85,56931.39,12790.77,4.450974414,8708.88,4906794,9885286.13,10.641126,7938.45,7972.55,7909.80,7963.20
1,2016-01-04,NIFTY,226345,16975875,13340.09,2546999,191024925,151788.40,165128.49,17179.35,9.612033633,11940.71,7498588,9724199.47,11.378364,7924.55,7937.55,7781.10,7791.30
2,2016-01-05,NIFTY,148669,11150175,8713.19,1945485,145911375,115647.05,124360.24,17657.69,7.042837427,14100.76,7865865,9752722.53,13.272642,7828.40,7831.20,7763.25,7784.65
3,2016-01-06,NIFTY,169347,12701025,9877.81,2131230,159842250,126551.65,136429.46,20110.22,6.784085903,12892.63,8997270,9722033.56,12.811711,7788.05,7800.95,7721.20,7741.00
4,2016-01-07,NIFTY,234900,17617500,13419.15,3324174,249313050,193468.48,206887.63,18650.32,11.09298017,11127.87,8259340,9480099.10,14.417342,7673.35,7674.95,7556.60,7568.30


In [2]:
import pandas as pd
import numpy as np

# 1. Load your main dataset
df = pd.read_csv('data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# 2. Add 'lot_size' COLUMN (Keeping our accurate lot size mapping)
conditions_lot = [
    df['Date'] <= '2021-06-30',
    (df['Date'] > '2021-06-30') & (df['Date'] <= '2024-04-25'),
    (df['Date'] > '2024-04-25') & (df['Date'] <= '2024-12-31'),
    (df['Date'] > '2024-12-31') & (df['Date'] <= '2026-01-05'),
    df['Date'] >= '2026-01-06'
]
choices_lot = [75, 50, 25, 75, 65]
df['lot_size'] = np.select(conditions_lot, choices_lot, default=np.nan)

# ---------------------------------------------------------
# 3. Add 'is_expiry' COLUMN (The Clean, Data-Driven Way!)
# ---------------------------------------------------------
# Load the 100% accurate official expiries we extracted from the raw ZIP files
expiries_df = pd.read_csv('nifty_actual_expiries.csv')

# Convert to a set of dates for fast lookup
actual_expiries = set(pd.to_datetime(expiries_df['ExpiryDate']).dt.date)

# Check if the trading date is in our official list of expiry dates
df['is_expiry'] = df['Date'].dt.date.isin(actual_expiries)

# Verify the output
print(df[['Date', 'lot_size', 'is_expiry']].head(10))
print(f"\nTotal Expiry Days Mapped: {df['is_expiry'].sum()}")


        Date  lot_size  is_expiry
0 2016-01-01      75.0      False
1 2016-01-04      75.0      False
2 2016-01-05      75.0      False
3 2016-01-06      75.0      False
4 2016-01-07      75.0      False
5 2016-01-08      75.0      False
6 2016-01-11      75.0      False
7 2016-01-12      75.0      False
8 2016-01-13      75.0      False
9 2016-01-14      75.0      False

Total Expiry Days Mapped: 429


In [3]:
df

,Date,Symbol,Futures Contracts,Futures Quantity,Futures Value (Rs. In Crs.),Options Contracts,Options Quantity,Options Value (Rs. In Crs.),Total Overall Value (Rs. In Crs.),Equity Traded Value (Rs. In Crs.),...,Equity Traded Quantity (in Lakhs),Equity Number of Trades,Equity Total Market Cap (Rs. Crs.),options/futures,open,high,low,close,lot_size,is_expiry
0,2016-01-01,NIFTY,81885,6141375,4890.54,859166,64437450,52040.85,56931.39,12790.77,...,8708.88,4906794,9885286.13,10.641126,7938.45,7972.55,7909.80,7963.20,75.0,False
1,2016-01-04,NIFTY,226345,16975875,13340.09,2546999,191024925,151788.40,165128.49,17179.35,...,11940.71,7498588,9724199.47,11.378364,7924.55,7937.55,7781.10,7791.30,75.0,False
2,2016-01-05,NIFTY,148669,11150175,8713.19,1945485,145911375,115647.05,124360.24,17657.69,...,14100.76,7865865,9752722.53,13.272642,7828.40,7831.20,7763.25,7784.65,75.0,False
3,2016-01-06,NIFTY,169347,12701025,9877.81,2131230,159842250,126551.65,136429.46,20110.22,...,12892.63,8997270,9722033.56,12.811711,7788.05,7800.95,7721.20,7741.00,75.0,False
4,2016-01-07,NIFTY,234900,17617500,13419.15,3324174,249313050,193468.48,206887.63,18650.32,...,11127.87,8259340,9480099.10,14.417342,7673.35,7674.95,7556.60,7568.30,75.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2594,2026-07-20,NIFTY,56686,3684590,8928.47,138857811,9025757715,21881242.72,21890171.19,107795.45,...,42777.09,34167108,48144567.04,2450.727025,24190.05,24266.10,24135.85,24238.50,65.0,False
2595,2026-07-21,NIFTY,50724,3297060,7984.95,327924200,21315073000,51611679.21,51619664.16,113350.33,...,42929.45,35355780,48343129.98,6463.619586,24216.05,24262.20,24135.65,24187.70,65.0,True
2596,2026-07-22,NIFTY,65044,4227860,10166.45,41738427,2712997755,6552201.63,6562368.08,111459.97,...,42662.67,35320601,47909998.20,644.492584,24150.45,24166.30,23961.40,23996.25,65.0,False
2597,2026-07-23,NIFTY,143453,9324445,22318.21,46396692,3015784980,7259056.13,7281374.34,111749.62,...,42307.80,35892377,47572808.85,325.252613,23904.80,23990.75,23807.20,23869.60,65.0,False


In [14]:
import altair as alt

# 1. Prepare the dataframe for Altair
pl = pd.DataFrame()

# We need the Date column for the X-axis
pl['Date'] = df['Date'] 

# Calculate the rolling mean and save it as a specific column
pl['focontracts_21d_ma'] = (df['Futures Quantity'] + df['Options Quantity']).rolling(21).mean()

# 2. Plot with Altair
chart = alt.Chart(pl).mark_line(color='#1f77b4').encode(
    x=alt.X('Date:T', title='Date'),
    y=alt.Y('focontracts_21d_ma:Q', title='FO Contracts (21-Day MA)'),
    tooltip=[
        alt.Tooltip('Date:T', title='Date', format='%Y-%m-%d'),
        alt.Tooltip('focontracts_21d_ma:Q', title='Contracts', format=',.0f')
    ]
).properties(
    title='21-Day Rolling Mean of Total Futures & Options Contracts',
    width=1200,
    height=600
).interactive()

# 3. Display the chart
chart


alt.Chart(...)

In [13]:
import altair as alt

# 1. Prepare the dataframe
pl = pd.DataFrame()
pl['Date'] = df['Date'] 

# Calculate the standardized daily volume
# Formula: (Total Contracts) * (Current Lot Size) / 75
total_contracts = df['Futures Contracts'] + df['Options Contracts']
pl['standardized_focontracts'] = total_contracts * 75 / df['lot_size'] 

# Calculate the 21-day rolling average of the standardized volume
pl['standardized_21d_ma'] = pl['standardized_focontracts'].rolling(21).mean()

# 2. Plot with Altair
chart = alt.Chart(pl).mark_line(color='#ff7f0e').encode(
    x=alt.X('Date:T', title='Date'),
    y=alt.Y('standardized_21d_ma:Q', title='Standardized Contracts (21-Day MA)'),
    tooltip=[
        alt.Tooltip('Date:T', title='Date', format='%Y-%m-%d'),
        alt.Tooltip('standardized_21d_ma:Q', title='Standardized Contracts', format=',.0f')
    ]
).properties(
    title='21-Day Rolling Mean of Standardized FO Contracts (Base Lot = 75)',
    width=1200,
    height=600
).interactive()

# 3. Display the chart
chart


alt.Chart(...)

In [16]:
import altair as alt
import pandas as pd

# 1. Create a Weekly identifier (Monday - Sunday)
df['YearWeek'] = df['Date'].dt.to_period('W')

# 2. Get the Total Options Volume on Expiry Days for each week
expiry_vols = df[df['is_expiry']].groupby('YearWeek')['Options Contracts'].sum()

# 3. Get the Mean Options Volume on Non-Expiry Days for each week
non_expiry_mean = df[~df['is_expiry']].groupby('YearWeek')['Options Contracts'].mean()

# 4. Calculate the Ratio
weekly_ratios = (expiry_vols / non_expiry_mean).reset_index()
weekly_ratios.columns = ['YearWeek', 'Relative_Expiry_Vol']

# Convert 'YearWeek' period back to a Timestamp (start of the week) for Altair plotting
weekly_ratios['Date'] = weekly_ratios['YearWeek'].dt.start_time

# FIX: Drop the Period column so Altair doesn't crash during JSON serialization!
weekly_ratios = weekly_ratios.drop(columns=['YearWeek'])

# Drop weeks that had no expiry (e.g., non-monthly weeks before weekly options started in 2019)
weekly_ratios = weekly_ratios.dropna().reset_index(drop=True)

# 5. Plot with Altair
chart = alt.Chart(weekly_ratios).mark_line(color='#2ca02c').encode(
    x=alt.X('Date:T', title='Date'),
    y=alt.Y('Relative_Expiry_Vol:Q', title='Expiry Vol / Non-Expiry Mean'),
    tooltip=[
        alt.Tooltip('Date:T', title='Week of', format='%Y-%m-%d'),
        alt.Tooltip('Relative_Expiry_Vol:Q', title='Relative Vol Ratio', format='.2f')
    ]
).properties(
    title='Relative Options Volume: Expiry Day vs Non-Expiry Days',
    width=800,
    height=400
).interactive()

# 6. Display the chart
chart


alt.Chart(...)

In [17]:
import altair as alt
import pandas as pd

# 1. Create a Weekly identifier (Monday - Sunday)
df['YearWeek'] = df['Date'].dt.to_period('W')

# 2. Get the Total Options Volume on Expiry Days for each week
expiry_vols = df[df['is_expiry']].groupby('YearWeek')['Options Contracts'].sum()

# 3. Get the Mean Options Volume on Non-Expiry Days for each week
non_expiry_mean = df[~df['is_expiry']].groupby('YearWeek')['Options Contracts'].mean()

# 4. Calculate the Ratio
weekly_ratios = (expiry_vols / non_expiry_mean).reset_index()
weekly_ratios.columns = ['YearWeek', 'Relative_Expiry_Vol']

# Convert 'YearWeek' period back to a Timestamp (start of the week) for Altair plotting
weekly_ratios['Date'] = weekly_ratios['YearWeek'].dt.start_time

# Drop the Period column so Altair doesn't crash during JSON serialization
weekly_ratios = weekly_ratios.drop(columns=['YearWeek'])

# Drop weeks that had no expiry 
weekly_ratios = weekly_ratios.dropna().reset_index(drop=True)

# 5. Plot with Altair (Notice point=True added here!)
chart = alt.Chart(weekly_ratios).mark_line(color='#2ca02c', point=True).encode(
    x=alt.X('Date:T', title='Date'),
    y=alt.Y('Relative_Expiry_Vol:Q', title='Expiry Vol / Non-Expiry Mean'),
    tooltip=[
        alt.Tooltip('Date:T', title='Week of', format='%Y-%m-%d'),
        alt.Tooltip('Relative_Expiry_Vol:Q', title='Relative Vol Ratio', format='.2f')
    ]
).properties(
    title='Relative Options Volume: Expiry Day vs Non-Expiry Days',
    width=800,
    height=400
).interactive()

# 6. Display the chart
chart


alt.Chart(...)

In [8]:
import altair as alt
import numpy as np
import pandas as pd

# 1. Prepare the dataframe and add phases (same as before)
pl = pd.DataFrame()
pl['Date'] = df['Date'] 

conditions = [
    pl['Date'] <= '2019-02-10',                                  
    (pl['Date'] >= '2019-02-11') & (pl['Date'] <= '2021-07-31'), 
    (pl['Date'] >= '2021-08-01') & (pl['Date'] <= '2023-09-03'), 
    (pl['Date'] >= '2023-09-04') & (pl['Date'] <= '2024-04-25'), 
    (pl['Date'] >= '2024-04-26') & (pl['Date'] <= '2024-11-19'), 
    (pl['Date'] >= '2024-11-20') & (pl['Date'] <= '2025-09-01'), 
    (pl['Date'] >= '2025-09-02') & (pl['Date'] <= '2026-01-05'), 
    pl['Date'] >= '2026-01-06'                                   
]

choices = [
    'I: Monthly expiry (Lot 75)', 
    'II: Weekly options begin', 
    'III: Weekly ecosystem expands', 
    'IV: Mon-Thu expiry sequence', 
    'V: Four weeklies (Lot 25)', 
    'VI: One weekly benchmark', 
    'VII: Tuesday expiry (Lot 75)', 
    'VIII: Recalibrated lot (Lot 65)'
]
pl['Phase'] = np.select(conditions, choices, default='Unknown')

# Calculate the standardized daily volume & MA
total_contracts = df['Futures Contracts'] + df['Options Contracts']
pl['standardized_focontracts'] = total_contracts * df['lot_size'] / 75
pl['standardized_21d_ma'] = pl['standardized_focontracts'].rolling(5).mean()

# ---------------------------------------------------------
# NEW: Create Layered Chart with Background Shading
# ---------------------------------------------------------

# 2. Extract the start and end dates of each phase to draw the background rectangles
bg_df = pl.groupby('Phase').agg(start=('Date', 'min'), end=('Date', 'max')).reset_index()

# 3. Create a shared color scale so the background exactly matches the line color
color_scale = alt.Scale(domain=choices, scheme='category10')

# 4. Layer 1: The background shading (low opacity)
background = alt.Chart(bg_df).mark_rect(opacity=0.15).encode(
    x='start:T',
    x2='end:T',
    color=alt.Color('Phase:N', scale=color_scale, legend=None) # Hide redundant legend
)

# 5. Layer 2: The line chart (thicker strokeWidth so it stands out)
lines = alt.Chart(pl).mark_line(strokeWidth=2.5).encode(
    x=alt.X('Date:T', title='Date'),
    y=alt.Y('standardized_21d_ma:Q', title='Standardized Contracts (21-Day MA)'),
    color=alt.Color('Phase:N', scale=color_scale, title='Regulatory Phase'),
    tooltip=[
        alt.Tooltip('Date:T', title='Date', format='%Y-%m-%d'),
        alt.Tooltip('Phase:N', title='Phase'),
        alt.Tooltip('standardized_21d_ma:Q', title='Standardized Contracts', format=',.0f')
    ]
)

# 6. Combine the layers using the '+' operator!
chart = (background + lines).properties(
    title='21-Day Rolling Mean of Standardized FO Contracts by Regulatory Phase',
    width=1200,
    height=600
).interactive()

# Display the chart
chart


alt.LayerChart(...)

In [21]:
import altair as alt
import pandas as pd
import numpy as np

# 1. Create a Weekly identifier (Monday - Sunday)
df['YearWeek'] = df['Date'].dt.to_period('W')

# 2. Get the Total Options Volume on Expiry Days for each week
expiry_vols = df[df['is_expiry']].groupby('YearWeek')['Options Contracts'].sum()

# 3. Get the Mean Options Volume on Non-Expiry Days for each week
non_expiry_mean = df[~df['is_expiry']].groupby('YearWeek')['Options Contracts'].mean()

# 4. Calculate the Ratio
weekly_ratios = (expiry_vols / non_expiry_mean).reset_index()
weekly_ratios.columns = ['YearWeek', 'Relative_Expiry_Vol']

# Convert 'YearWeek' period back to a Timestamp (start of the week) for Altair plotting
weekly_ratios['Date'] = weekly_ratios['YearWeek'].dt.start_time

# Drop the Period column so Altair doesn't crash during JSON serialization
weekly_ratios = weekly_ratios.drop(columns=['YearWeek'])

# Drop weeks that had no expiry 
weekly_ratios = weekly_ratios.dropna().reset_index(drop=True)

# ---------------------------------------------------------
# NEW: Add Regulatory Phases to the Weekly Data
# ---------------------------------------------------------
conditions = [
    weekly_ratios['Date'] <= '2019-02-10',                                  
    (weekly_ratios['Date'] >= '2019-02-11') & (weekly_ratios['Date'] <= '2021-07-31'), 
    (weekly_ratios['Date'] >= '2021-08-01') & (weekly_ratios['Date'] <= '2023-09-03'), 
    (weekly_ratios['Date'] >= '2023-09-04') & (weekly_ratios['Date'] <= '2024-04-25'), 
    (weekly_ratios['Date'] >= '2024-04-26') & (weekly_ratios['Date'] <= '2024-11-19'), 
    (weekly_ratios['Date'] >= '2024-11-20') & (weekly_ratios['Date'] <= '2025-09-01'), 
    (weekly_ratios['Date'] >= '2025-09-02') & (weekly_ratios['Date'] <= '2026-01-05'), 
    weekly_ratios['Date'] >= '2026-01-06'                                   
]

choices = [
    'I: Monthly expiry (Lot 75)', 
    'II: Weekly options begin', 
    'III: Weekly ecosystem expands', 
    'IV: Mon-Thu expiry sequence', 
    'V: Four weeklies (Lot 25)', 
    'VI: One weekly benchmark', 
    'VII: Tuesday expiry (Lot 75)', 
    'VIII: Recalibrated lot (Lot 65)'
]
weekly_ratios['Phase'] = np.select(conditions, choices, default='Unknown')

# Extract start and end dates for the background shading
bg_df = weekly_ratios.groupby('Phase').agg(start=('Date', 'min'), end=('Date', 'max')).reset_index()

# Define the shared color scale
color_scale = alt.Scale(domain=choices, scheme='category10')

# 5. Layer 1: Background Shading
background = alt.Chart(bg_df).mark_rect(opacity=0.15).encode(
    x='start:T',
    x2='end:T',
    color=alt.Color('Phase:N', scale=color_scale, legend=None)
)

# 6. Layer 2: The Line Chart (with points!)
lines = alt.Chart(weekly_ratios).mark_line(point=True, strokeWidth=2.0).encode(
    x=alt.X('Date:T', title='Date'),
    y=alt.Y('Relative_Expiry_Vol:Q', title='Expiry Vol / Non-Expiry Mean'),
    color=alt.Color('Phase:N', scale=color_scale, title='Regulatory Phase'),
    tooltip=[
        alt.Tooltip('Date:T', title='Week of', format='%Y-%m-%d'),
        alt.Tooltip('Phase:N', title='Phase'),
        alt.Tooltip('Relative_Expiry_Vol:Q', title='Relative Vol Ratio', format='.2f')
    ]
)

# 7. Combine the layers
chart = (background + lines).properties(
    title='Relative Options Volume: Expiry Day vs Non-Expiry Days by Phase',
    width=1200,
    height=600
).interactive()

# Display the chart
chart


alt.LayerChart(...)